In [1]:
# Load env variables and create client
import base64
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-5"

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=None, stop_sequences=None, tools=None):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
    }

    if system:
        params["system"] = system

    if temperature is not None:
        params["temperature"] = temperature

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    if tools:
        params["tools"] = tools

    return client.messages.create(**params)


def text_from_message(message):
    return "\n".join(block.text for block in message.content if block.type == "text")

In [3]:
# Fire risk assessment prompt
prompt = """
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
   - The largest roofed structure 
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)
   Describe the residence's location relative to property boundaries and other features.

2. Tree overhang analysis: Examine all trees near the primary residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)
   - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
   - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
   - Proximity to chimneys, vents, or other roof openings if visible
   - Areas where branches create a "bridge" between wildland vegetation and the structure
   
4. Defensible space identification: Assess the property's overall vegetative structure:
   - Identify if trees connect to form a continuous canopy over or near the home
   - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the structure
   - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies
   - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple points of vulnerability
   - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation against structure, numerous ember catch points, limited defensible space

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numeric Fire Risk Rating (1-4) with a brief justification.
"""

In [4]:
# Encode the satellite image as base64 and send it as an image block
# placed before the text prompt — image-first ordering gives best results
with open("images/prop7.png", "rb") as f:
    image_data = base64.standard_b64encode(f.read()).decode("utf-8")

messages = []

add_user_message(
    messages,
    [
        {
            "type": "image",
            "source": {
                "type": "base64",
                "media_type": "image/png",
                "data": image_data,
            },
        },
        {"type": "text", "text": prompt},
    ],
)

response = chat(messages)
print(text_from_message(response))

1. **Residence identification:** The primary residence is a moderately sized, irregularly-shaped (multi-wing) structure located centrally in the image, surrounded on all sides by dense tree cover with no visible driveway or clear property boundary lines distinguishable from the forest.

2. **Tree overhang analysis:** Dense tree canopies press directly against and over multiple sections of the roof, particularly on the west, south, and east edges, with an estimated 25-50% of the roof affected by overhanging branches, especially heavy at the lower (southern) wing.

3. **Fire risk assessment:** The overhanging branches create direct bridging points between wildland vegetation and the roofline, with dark shadow patterns indicating branches in close contact with roof edges, posing ember-catch risk at valleys and roof-to-tree contact zones; no chimney or vent details are clearly visible but roof junctions are obscured by foliage.

4. **Defensible space identification:** The surrounding tree 

In [5]:
# Same rubric on a different property — a reusable prompt makes the
# assessments comparable across images
with open("images/prop1.png", "rb") as f:
    image_data = base64.standard_b64encode(f.read()).decode("utf-8")

messages = []

add_user_message(
    messages,
    [
        {
            "type": "image",
            "source": {
                "type": "base64",
                "media_type": "image/png",
                "data": image_data,
            },
        },
        {"type": "text", "text": prompt},
    ],
)

response = chat(messages)
print(text_from_message(response))

# Property Fire Risk Analysis

**1. Residence Identification:** The primary residence is a single structure with a gray/tan roof located in the center-left portion of the image, surrounded almost entirely by dense tree canopy with only small visible sections of the roofline (peak and portions of the roof plane) visible through gaps in vegetation.

**2. Tree Overhang Analysis:** Multiple large deciduous/coniferous trees directly overhang the structure, with dense canopy covering an estimated 50-75% of the roof area, particularly heavy overhang visible on the lower-left and bottom portions of the roof where trees appear to make direct contact with the roofline.

**3. Fire Risk Assessment:** The extensive canopy contact creates significant ember catch potential along the roofline and in valleys where branches meet the roof surface; no chimney or vents are clearly visible in the exposed roof sections, but the dense overhang itself represents a direct fuel bridge from tree crowns to the str